# 24 · PyTorch en pratique 2 : améliorer, chercher, CNN et RNN — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/chapitres/ch24_pytorch_2/05_solutions.ipynb)

Les réponses des exercices, exécutées. Les démarches détaillées (le *pourquoi*, les erreurs fréquentes, les variantes) sont dans `05_solutions.md`. Les cellules marquées `answer` enregistrent les réponses vérifiées par `wb.check` (`tools/build_answers.py`).

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | 24.Q1 à 24.Q12, 24.R1 à 24.R3, 24.1 à 24.5, 24.7 | vérifier tes réponses courtes | 🧠 🔁 ✏️ 🧮 | ★ à ★★ | |
| A | 24.11 à 24.15 | améliorer un MLP sur Fashion-MNIST : taille de batch, quatre MLP, dropout et max-norm, validation au-dessus du train, journal d'expériences (*prochaine session de génération*) | | | |
| B | 24.16 à 24.18 | un réseau PyTorch en estimateur scikit-learn : validation croisée, grille, hasard et zoom (*prochaine session de génération*) | | | |
| C | 24.19 à 24.24 | des CNN sur CIFAR-10 : un 🐛 de traduction, du simple au profond, batchnorm et dropout, augmentation, images synthétiques (*prochaine session de génération*) | | | |
| D | 24.25 à 24.27 | des séries et un LSTM : sommes de sinus, fenêtre, profondeur et données, sorties et état porté sur les taches solaires (*prochaine session de génération*) | | | |
| E | 24.28 à 24.31 | au-delà de nn.Sequential : branches et blocs résiduels, précision mixte et compilation, génération de texte, défi CIFAR-10 frugal (*prochaine session de génération*) | | | |

In [1]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mylearn = wb.load_mylearn("ref")  # reference implementation

🧪 Deep Learning Workbook : environnement
   Plateforme : Local (Linux x86_64, 2 CPU)
   Python 3.13.13 | numpy 2.1.3 | pandas 2.2.3 | matplotlib 3.10.0 | scikit-learn 1.6.1 | scipy 1.16.3 | torch 2.11.0+cpu | torchvision 0.26.0+cpu
   Device : cpu | Graine : 42 | FAST_MODE : True


## Objectifs et rappel express

- Améliorer un réseau par expériences : un hyperparamètre à la fois, dropout et max-norm, un journal de chaque run.
- Régler un réseau PyTorch avec scikit-learn : estimateur maison, `Pipeline`, validation croisée, recherches en grille et au hasard.
- Construire des CNN sur CIFAR-10 (stride, batchnorm, augmentation, blocs résiduels), préparer une série pour un LSTM, lire les sorties de `nn.LSTM`, générer du texte.

**Rappel express.** Mises à jour par epoch : $\lceil N/B \rceil$ ; bruit d'un mini-batch en $1/\sqrt{B}$. Dropout inversé : $\tilde h = m\,h/(1-p)$. Max-norm : $\mathbf{w} \leftarrow \mathbf{w}\min(1, c/\lVert\mathbf{w}\rVert)$. Coût d'une grille : $K \times \prod_j |V_j|$. Convolution : $C_{out}(C_{in}k^2 + 1)$ paramètres, $H_{out}W_{out}C_{out}C_{in}k^2$ MAC. `nn.LSTM(batch_first=True)` : `output` $(N, T, H)$, `h_n` $(L, N, H)$. Bloc résiduel : $y = F(x) + x$.

## Partie 0 · Vérifier tes réponses courtes (quiz, rappels, ✏️ 24.1 à 24.5 et 🧮 24.7)

Fais d'abord les quiz, les rappels et les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse courte ici : **la valeur** que tu as trouvée, pas l'expression Python, sinon tu ne vérifies rien. Un nombre : `42` ou `0.3` (en Python, le séparateur décimal est un **point** ; `0,3` sans guillemets serait un couple de deux nombres) ; un vrai ou faux : `True` ou `False` ; un choix : la lettre seule, entre guillemets (`"F"`) ; plusieurs lettres : entre guillemets (`"AF"`) ; une forme : entre guillemets (`"(3, 5, 1)"`). Arrondis comme l'énoncé le demande, et seulement à la fin du calcul. Les réponses pas encore remplies affichent ⏳. Les questions « dans ta copie », la dérivation ∂ 24.6, la réflexion (🗣️ 24.8, ⚖️ 24.9, 📄 24.10) et l'entretien se corrigent avec `05_solutions.md`.

In [2]:
# The short answers only use the numbers of their statements (02_exercices.md)
import math

import numpy as np


def softmax(z):
    z = np.asarray(z, dtype=float)
    e = np.exp(z - z.max())
    return e / e.sum()


def conv_params(c_in, c_out, k):
    """Parameters of nn.Conv2d(c_in, c_out, k): weights and biases."""
    return c_out * (c_in * k * k + 1)


def dense(n_in, n_out):
    """Parameters of nn.Linear(n_in, n_out): weights and biases."""
    return (n_in + 1) * n_out


def conv_mac(h_out, w_out, c_out, c_in, k):
    """Multiply-accumulates of one convolution, for one image."""
    return h_out * w_out * c_out * c_in * k * k


def lstm_layer_params(d_in, h):
    """Parameters of one PyTorch LSTM layer: 4 blocks of weights, two bias vectors."""
    return 4 * h * (d_in + h) + 2 * 4 * h


# Recall 24.R3: logits of three tokens
Z_R3 = np.array([2.0, 1.0, 0.0])

# Ex 24.2: the book's pooling CNN on MNIST (fig. 24.31), in PyTorch
P_CONV1, P_CONV2 = conv_params(1, 30, 5), conv_params(30, 16, 3)
N_FLAT = 16 * 7 * 7
P_DENSE = dense(N_FLAT, 128) + dense(128, 64) + dense(64, 10)
P_TOTAL = P_CONV1 + P_CONV2 + P_DENSE

# Ex 24.3: multiply-accumulates of one image, pooling version and stride version
MAC_POOL = conv_mac(28, 28, 30, 1, 5) + conv_mac(14, 14, 16, 30, 3)
MAC_STRIDE = conv_mac(14, 14, 30, 1, 5) + conv_mac(7, 7, 16, 30, 3)
MAC_DENSE = 784 * 128 + 128 * 64 + 64 * 10

# Ex 24.7: a small CNN, 25 million MAC per image, 20 epochs on 50 000 images
OPS_247 = 25e6 * 2 * 3 * 50_000 * 20
CPU_MIN_247 = OPS_247 / 50e9 / 60

**24.Q1 — Compter les hyperparamètres et n'en changer qu'un**

In [3]:
wb.record("24.Q1a", "BCE", mistakes={"les poids de la première couche sont calculés par l'entraînement, pas choisis": "ABCE",
          "les biais sont appris comme les poids": "BCDE",
          "l'état d'Adam est calculé pendant l'entraînement, pas choisi": "BCEF",
          "le taux de dropout se choisit avant l'entraînement": "BC",
          "le learning rate se choisit avant l'entraînement": "BE"})
wb.record("24.Q1b", 3 ** 6, mistakes={"chaque hyperparamètre multiplie le nombre de combinaisons, il ne l'additionne pas": 18,
          "3 valeurs pour chacun des 6 : 3 puissance 6, pas 6 puissance 3": 6 ** 3})
wb.record("24.Q1c", False, mistakes={"garder la valeur par défaut, est-ce un choix ou non ?": True})
wb.record("24.Q1d", False, mistakes={"si le meilleur réglage de A dépend de B, régler A une seule fois suffit-il ?": True})
wb.record("24.Q1e", "C", mistakes={"les objectifs se contredisent souvent : on ne peut pas tous les servir à la fois": "A",
          "le livre cherche un réglage assez bon, pas le meilleur exact": "B",
          "changer plusieurs choses à la fois empêche de savoir laquelle a agi": "D"})

WB_ANSWER {"hash": "7535440018c570516dc1bd2d665b20129a6cf67f971c2627edfa80442269bb91", "id": "24.Q1a", "kind": "str", "mistakes": {"03d9a9437cd14de6a9c199df4accda5bc694a8b0b4c37f31d2cd4c290a1b9f85": "les poids de la première couche sont calculés par l'entraînement, pas choisis", "2e16ebd3eec8d21018013bbab41f6f0962c529ebaea29f9d8923b715280fcebe": "le learning rate se choisit avant l'entraînement", "31b8b73e7d2896d332bb257b650c3642de20de06d25fa3a49679424ea1bc3ee8": "les biais sont appris comme les poids", "4db1c0c81cd5b12d7e2d0795a66eae365d94ba5aa67f76f61fe90d256394885e": "l'état d'Adam est calculé pendant l'entraînement, pas choisi", "f8fdba40f270717aa218f2bf8011d9024a48bca8f8d3ed1a5fc2f780f9b80e08": "le taux de dropout se choisit avant l'entraînement"}}
📝 Ex 24.Q1a : réponse enregistrée (str).
WB_ANSWER {"hash": "cfe8bddf6c44e3a5cb000218c600406188d31b646c6d7acb3acc570d947f2cf3", "id": "24.Q1b", "kind": "int", "magnitude": 2, "mistakes": {"8f0b963916da1094bfafaf59e8b265a5be66b53ec1a11d0

**24.Q2 — Taille de batch : bruit, sur-apprentissage et temps**

In [4]:
wb.record("24.Q2a", math.ceil(50_000 / 64), fractional="un nombre de mises à jour est entier : le dernier mini-batch, incomplet, compte aussi",
          mistakes={"arrondis au-dessus : le dernier mini-batch, incomplet, compte aussi": 50_000 // 64,
          "une mise à jour par mini-batch : divise N par la taille du mini-batch": 50_000})
wb.record("24.Q2b", math.ceil(50_000 / 256), fractional="un nombre de mises à jour est entier : le dernier mini-batch, incomplet, compte aussi",
          mistakes={"arrondis au-dessus : le dernier mini-batch, incomplet, compte aussi": 50_000 // 256,
          "une mise à jour par mini-batch : divise N par la taille du mini-batch": 50_000})
wb.record("24.Q2c", math.sqrt(400) / math.sqrt(16), decimals=1, mistakes={"l'écart-type varie comme 1/√B, pas comme 1/B": 25, "dans quel sens varie le bruit quand le batch rapetisse ?": 0.2})
wb.record("24.Q2d", False, mistakes={"compare le nombre de pas par epoch et le bruit de chaque pas dans les deux cas": True})
wb.record("24.Q2e", "A", mistakes={"plus de mini-batches par epoch, c'est plus de pas, chacun avec son coût fixe": "B",
          "relis la fig. 24.5 : la durée dépend bien de la taille de batch": "C"})

WB_ANSWER {"fractional": "un nombre de mises à jour est entier : le dernier mini-batch, incomplet, compte aussi", "hash": "f36d54ac5ebdc0f24e01b359b7c108befc6205b5a0d681c70d51a194fc1f0144", "id": "24.Q2a", "kind": "int", "magnitude": 2, "mistakes": {"2f5cf3d3ce2fccc922438ce0b65558c5dba93f0ea1bb4a47bb14bd919df9bfc2": "arrondis au-dessus : le dernier mini-batch, incomplet, compte aussi", "4c85b22a0ca8a78ef1822be236a624f77d5a32c279c0b655ea371b92d8ea1dc4": "une mise à jour par mini-batch : divise N par la taille du mini-batch"}, "sign": 1}
📝 Ex 24.Q2a : réponse enregistrée (int).
WB_ANSWER {"fractional": "un nombre de mises à jour est entier : le dernier mini-batch, incomplet, compte aussi", "hash": "daccf03da18216b2db23129333e604f6fa387883e8c95b4857bce32270584b69", "id": "24.Q2b", "kind": "int", "magnitude": 2, "mistakes": {"1047300144c433efc1e12723377537cf46e61a197f1a60852923d8c654c8457c": "arrondis au-dessus : le dernier mini-batch, incomplet, compte aussi", "86b618f51b64a6360d4accdbd68

**24.Q3 — Plus de neurones, moins de neurones, dropout**

In [5]:
wb.record("24.Q3a", "D", mistakes={"plus de neurones, c'est plus de capacité, pas moins": "A",
          "relis ce que montre la fig. 24.7 sur la loss de validation": "B",
          "un réseau plus grand apprend le train au moins aussi bien": "C"})
wb.record("24.Q3b", 0.3, decimals=1, mistakes={"p est la probabilité de mettre à zéro, pas de garder": 0.7, "une fraction entre 0 et 1, pas un pourcentage": 30})
wb.record("24.Q3c", "C", mistakes={"relis l'encadré « Le livre, corrigé » : le masque ne dure pas une epoch": "A",
          "chaque exemple du mini-batch a son propre masque": "B",
          "un masque fixe éteindrait toujours les mêmes neurones : ce serait un réseau plus petit": "D"})
wb.record("24.Q3d", 2.0 / (1 - 0.2), decimals=1, mistakes={"le dropout inversé divise les valeurs gardées par 1 − p : il ne les laisse pas telles quelles": 2.0,
          "on divise par 1 − p, on ne multiplie pas": 1.6, "on divise par 1 − p, pas par p": 10.0})
wb.record("24.Q3e", False, mistakes={"c'est le dropout d'origine qui multipliait par 1 − p en évaluation ; que fait le dropout inversé ?": True})

WB_ANSWER {"hash": "5534a8ecc770304a12a6d54502633ca979e563824cd8000348be1d74ff7b01fd", "id": "24.Q3a", "kind": "str", "mistakes": {"8a6d788ed8c86a40cc815ab864e5d3e03713c0df51a3d18388001957a27c1073": "un réseau plus grand apprend le train au moins aussi bien", "b6755eb63a1106d0d6e26bb5775bef9a1e157267df730efb97329d7c5213961e": "plus de neurones, c'est plus de capacité, pas moins", "c34a38ebe83cd11b46710d1b769a380e66582f1fd1295d76ea0a69bd17c68fce": "relis ce que montre la fig. 24.7 sur la loss de validation"}}
📝 Ex 24.Q3a : réponse enregistrée (str).
WB_ANSWER {"decimals": 1, "hash": "1168c2a3d51c97cf3da38d69f398e11f9d0e897321fb1e56a03ee26203119638", "hash_coarse": "335770dcbca8656f3015f3ebdda0b64269768606dd7c865893924b1e05602e5a", "id": "24.Q3b", "kind": "float", "magnitude": -1, "mistakes": {"b6f7e01c95f4fa7e81af51fe0f6d273975ee389288143c6dc85382d99581c298": "une fraction entre 0 et 1, pas un pourcentage", "fa36ba27e2989adcbf5e7fe0d9863fdd084bd506212cb4ba8918357477c29319": "p est la pr

**24.Q4 — Valider un réseau : coût, jeu non représentatif, normalisation dans chaque fold**

In [6]:
wb.record("24.Q4a", 5 * 40, mistakes={"chaque fold est un entraînement complet": 40,
          "la validation croisée seule, sans réentraînement final sur toutes les données": 6 * 40})
wb.record("24.Q4b", "A", mistakes={"2 000 radios sur 10 000, la taille n'est pas le problème : regarde leur provenance": "B",
          "rien ne fait passer les labels dans les données : regarde de quel hôpital viennent ces radios": "C",
          "regarde de quel hôpital viennent les 2 000 dernières radios": "D"})
wb.record("24.Q4c", "D", mistakes={"le pipeline ne fait rien gagner en vitesse": "A",
          "cross_val_score accepte n'importe quelles données": "B",
          "le scaler s'applique aux features, pas aux labels": "C"})
wb.record("24.Q4d", True, mistakes={"relis ce que scikit-learn fait d'un cv entier quand l'estimateur est un classifieur (ch. 15)": False})
wb.record("24.Q4e", "C", mistakes={"relis le score du listing 24.16 et compare-le à celui du listing 24.12": "A",
          "les deux scores du livre sont presque égaux": "B"})

WB_ANSWER {"hash": "a0fa4700d33f7434034892b49cc94b46b1ec08e09c6beb707cf4479b90512ef9", "id": "24.Q4a", "kind": "int", "magnitude": 2, "mistakes": {"85a75c9b192e8df0ca09ca6474f5376e54ad6fd49b0172478c506c36384aa1e0": "la validation croisée seule, sans réentraînement final sur toutes les données", "8d42e255554e043c5027c90c956f6d330a151a34227091f072adbca6053de070": "chaque fold est un entraînement complet"}, "sign": 1}
📝 Ex 24.Q4a : réponse enregistrée (int).
WB_ANSWER {"hash": "1ec9f848ab537260df9e7ee1fc4315105803b841f3877cbd6d047668b7d01839", "id": "24.Q4b", "kind": "str", "mistakes": {"45b7325fdef7e9bfc2a9e99fe58cb5a416189c6e384c4beec52572bc397f3215": "rien ne fait passer les labels dans les données : regarde de quel hôpital viennent ces radios", "8e82c92003c0b1dc2928bbce36732fa41544aaaf1cba5530fe9cf87cab28fe88": "2 000 radios sur 10 000, la taille n'est pas le problème : regarde leur provenance", "a6891d4766da68f29a7adabf18491024a39b33caea4764b007dc668f3c3b0334": "regarde de quel hôpit

**24.Q5 — 🕰️ KerasClassifier, grille, hasard ou zoom : chercher des hyperparamètres**

In [7]:
wb.record("24.Q5a", "C", mistakes={"KerasClassifier a été retiré de Keras et n'a jamais enveloppé de réseau PyTorch": "A",
          "MLPClassifier est le réseau de scikit-learn, pas ton nn.Module": "B",
          "un estimateur n'est qu'un objet qui suit quelques conventions": "D"})
wb.record("24.Q5b", 5, mistakes={"c'était la valeur par défaut en 2017, dans le livre ; relis l'encadré 🕰️": 3,
          "pense à la valeur par défaut, pas à un choix courant": 10,
          "c'est un réglage par défaut de scikit-learn, pas un calcul : relis l'encadré 🕰️ de la recherche": 4})
wb.record("24.Q5c", 2 * 3 * 2 * 4, mistakes={"chaque candidat est entraîné une fois par fold": 2 * 3 * 2,
          "multiplie les nombres de valeurs, ne les additionne pas": (2 + 3 + 2) * 4,
          "relis la question : on demande les entraînements de la validation croisée seulement": 2 * 3 * 2 * 4 + 1})
wb.record("24.Q5d", 2 * 3 * 2 * 4 + 1, mistakes={"le réentraînement final s'ajoute une fois, sur toutes les données": 2 * 3 * 2 * 4,
          "un seul réentraînement final, pas un par candidat": 2 * 3 * 2 * 5})
wb.record("24.Q5e", "D", mistakes={"il s'agit de la finesse de la recherche, pas de la taille des images": "A",
          "le hasard est une autre méthode de recherche": "B",
          "il s'agit de la finesse de la recherche, pas du matériel": "C"})

WB_ANSWER {"hash": "5e7ab869e8be8f0593b2ee417807bce05ecb5a4536dd75d48ba85be632021294", "id": "24.Q5a", "kind": "str", "mistakes": {"27e5eae3b8a1ffa10ae390994cba975d61f5f9e1cdf195755f2fbe84b0e9847d": "KerasClassifier a été retiré de Keras et n'a jamais enveloppé de réseau PyTorch", "999c51a00968216436ba665bf3a069349b3f7bd8b46b3827b4adf0dfaddfe382": "MLPClassifier est le réseau de scikit-learn, pas ton nn.Module", "dba051a63e7bc2bffdf3e07ecadef54b33922a90cd870942ef8c0233fbb2dc1f": "un estimateur n'est qu'un objet qui suit quelques conventions"}}
📝 Ex 24.Q5a : réponse enregistrée (str).
WB_ANSWER {"hash": "36ba8da1d983443969c0892a75b6e23a4f2edaab7aca5c621760f0212fe3bbb6", "id": "24.Q5b", "kind": "int", "magnitude": 0, "mistakes": {"100fcf20a5a57e94c7692a1d116034205a6c36da93cb6dc3f360675e312faa44": "pense à la valeur par défaut, pas à un choix courant", "c874455b06bfe62250461e5b1525af243026a551b45052c68bce26bc407a7225": "c'est un réglage par défaut de scikit-learn, pas un calcul : relis l'

**24.Q6 — 🕰️ Couches utilitaires de Keras et leurs équivalents PyTorch**

In [8]:
wb.record("24.Q6a", "C", mistakes={"cette couche agrandit l'image : elle ne retire rien": "A",
          "elle répète des valeurs existantes ; elle n'ajoute pas de zéros": "B",
          "elle agrandit l'image ; un pooling la rétrécit": "D",
          "elle garde la forme d'une image": "E"})
wb.record("24.Q6b", "A", mistakes={"le recadrage retire des bords, il n'en ajoute pas": "B",
          "le recadrage rétrécit l'image": "C",
          "le recadrage garde les valeurs telles quelles, sans les résumer": "D",
          "le recadrage garde la forme d'une image": "E"})
wb.record("24.Q6c", "B", mistakes={"ZeroPadding2D ajoute des bords": "A",
          "ZeroPadding2D ajoute des zéros, il ne répète pas les valeurs": "C",
          "ZeroPadding2D agrandit l'image": "D",
          "ZeroPadding2D garde la forme d'une image": "E"})
wb.record("24.Q6d", "D", mistakes={"un pooling résume des blocs, il ne retire pas seulement les bords": "A",
          "un pooling rétrécit l'image": "B", "un pooling rétrécit l'image, il ne l'agrandit pas": "C",
          "un pooling garde la forme d'une image": "E"})
wb.record("24.Q6e", False, mistakes={"relis quand ce bruit est actif dans le tableau du §24.4.1": True})
wb.record("24.Q6f", "C", mistakes={"relis l'encadré « Le livre, corrigé » : ce ne sont pas les poids": "A",
          "la batchnorm agit pendant la passe avant, pas sur les gradients": "B",
          "la batchnorm agit dans le réseau, sur les sorties d'une couche": "D"})

WB_ANSWER {"hash": "1b38ca10065e46e32785cf96f27ac250941907cb960c8a2c242658a2347672b1", "id": "24.Q6a", "kind": "str", "mistakes": {"4bd3cc68cee70e6a4663dc22543d75ab6fdcf76629d35a1593c02b1e157188ca": "elle agrandit l'image ; un pooling la rétrécit", "d5653126b2333737d1243c64e8003c1029b815b918965a95e3221abb5752c36a": "elle répète des valeurs existantes ; elle n'ajoute pas de zéros", "f66402f7b893ee112d63386cdfa1fa0271c356d3bf84742b2b8ef071720d2206": "cette couche agrandit l'image : elle ne retire rien", "fc59435a310abe9169dcfecdf4c9897693758d08654ee29b4a2b31d03450c20b": "elle garde la forme d'une image"}}
📝 Ex 24.Q6a : réponse enregistrée (str).
WB_ANSWER {"hash": "545e0dca3b2d392a31c3939b2889156ff187ab9398fdb746e5d94a957d972530", "id": "24.Q6b", "kind": "str", "mistakes": {"288a7c8dadf64a3398bf7df6b65c1d977c745b496046e0a7a08894bc6f30371d": "le recadrage retire des bords, il n'en ajoute pas", "28ea71de789f65f40244559ed93e0b0226ed9876e0e19e1baf50bc2951e4a6da": "le recadrage garde la forme

**24.Q7 — Préparer des images pour nn.Conv2d**

In [9]:
wb.record("24.Q7a", "(16, 3, 32, 32)", mistakes={"PyTorch veut les canaux juste après l'axe des exemples": "(16, 32, 32, 3)",
          "n'oublie pas l'axe des exemples": "(3, 32, 32)"})
wb.record("24.Q7b", "D", mistakes={"reshape garde l'ordre de lecture : il mélangerait les pixels et les canaux": "A",
          "la forme est bonne, mais transpose(1, 3) échange aussi la hauteur et la largeur": "B",
          "écris la forme obtenue axe par axe : l'axe des canaux n'arrive pas en position 1": "C"})
wb.record("24.Q7c", "(16, 12, 32, 32)", mistakes={"padding=\"same\" garde la hauteur et la largeur": "(16, 12, 28, 28)",
          "les canaux de sortie sont le nombre de filtres": "(16, 3, 32, 32)"})
wb.record("24.Q7d", False, mistakes={"relis la première des trois différences avec Keras au §24.4.3": True})
wb.record("24.Q7e", 51 / 127.5 - 1, decimals=1, mistakes={"x / 127,5 − 1 : n'oublie pas de retrancher 1": 0.4, "x / 127,5 − 1, pas x / 255": 0.2})
wb.record("24.Q7f", False, mistakes={"relis la deuxième des trois différences avec Keras au §24.4.3": True})

WB_ANSWER {"hash": "d7667eceef71615e09d0a17a03d7e36d54f71dd4ffed6b220a0d791d0829d8db", "id": "24.Q7a", "kind": "str", "mistakes": {"bf6f4fc11c27fac4312d734094b8f2de8110cbdb9292159e6afab8a5dd6f65de": "n'oublie pas l'axe des exemples", "eda772f4cfcd13ccaa0cc2f1fce352cf2b225b25b6ec8e0cd17226d90d6bd8ec": "PyTorch veut les canaux juste après l'axe des exemples"}}
📝 Ex 24.Q7a : réponse enregistrée (str).
WB_ANSWER {"hash": "0144b558a93711fd0494bed3e3e604793a966f5e802d0ce8144311f1a316de66", "id": "24.Q7b", "kind": "str", "mistakes": {"00ad01a6913171be1161c46c3cdf5821ee144df74599e063a18cc728209ef8ef": "écris la forme obtenue axe par axe : l'axe des canaux n'arrive pas en position 1", "0964626ad6f577dcd669d5b2b327a0b85fafe5bc5c6be3529c1849280b8b5dc0": "reshape garde l'ordre de lecture : il mélangerait les pixels et les canaux", "c0288261ec28024058a35b3a9135b1262584ea3beb1e7e3c0be3cd8b190e7fde": "la forme est bonne, mais transpose(1, 3) échange aussi la hauteur et la largeur"}}
📝 Ex 24.Q7b : rép

**24.Q8 — Augmentation et données synthétiques : quelles transformations sont plausibles ?**

In [10]:
wb.record("24.Q8a", "B", mistakes={"la validation doit ressembler aux données réelles : on ne l'augmente pas": "A",
          "le test doit ressembler aux données réelles": "C",
          "l'augmentation fabrique des exemples d'entraînement": "D"})
wb.record("24.Q8b", False, mistakes={"quand le Dataset tire-t-il ses transformations ?": True})
wb.record("24.Q8c", "C", mistakes={"trop forte, une augmentation empêche le modèle d'apprendre": "A",
          "une augmentation garde le label de l'image": "B",
          "on n'augmente jamais le test": "D"})
wb.record("24.Q8d", "D", mistakes={"le générateur fabrique les batches à la demande : la mémoire n'est pas en jeu": "A",
          "le learning rate est un réglage de l'optimiseur": "B",
          "le mélange n'a rien à voir avec la fin d'une epoch": "C"})
wb.record("24.Q8e", "B", mistakes={"un IterableDataset n'a pas de __len__": "A", "la taille du batch est un réglage du DataLoader": "C"})
wb.record("24.Q8f", True, mistakes={"le modèle revoit-il deux fois le même exemple ?": False})

WB_ANSWER {"hash": "37a134ab97de18309323e1b8063fb349bb05cc3e582d341763e0cd3c44179e0e", "id": "24.Q8a", "kind": "str", "mistakes": {"38c0bb5c71130575d7e8b905307bccc1c4c604ddcfb7736d16fe4425dd72d754": "le test doit ressembler aux données réelles", "a1f75f75ffffca54faed35827191e1baaf1e609e781a8d8b00cac5ab4e23431a": "la validation doit ressembler aux données réelles : on ne l'augmente pas", "d18f42494ad6ca6f4bb66948f223df94e09ff7af4e5482b663870384deaf530f": "l'augmentation fabrique des exemples d'entraînement"}}
📝 Ex 24.Q8a : réponse enregistrée (str).
WB_ANSWER {"hash": "c3cb44add84a7a7f34be810227a2f4aad2e6787676920f828b1a6b18aa62096b", "id": "24.Q8b", "kind": "bool", "mistakes": {"811101af675316e34b68fb4b80611bdbb9b55f4cebe4d7e169578b4e1a3434f2": "quand le Dataset tire-t-il ses transformations ?"}}
📝 Ex 24.Q8b : réponse enregistrée (bool).
WB_ANSWER {"hash": "bc1babcd277d7e6213a4cc6a260ab6569b8eb79e3dc7fa5dc77dd20171a17395", "id": "24.Q8c", "kind": "str", "mistakes": {"33c03669a4708aa5e6

**24.Q9 — Motifs conv-conv-pool, stride contre pooling, batchnorm contre dropout**

In [11]:
wb.record("24.Q9a", "C", mistakes={"relis le listing 24.44 : la convolution est écrite sans activation": "A",
          "la batchnorm normalise la sortie de la convolution": "B"})
wb.record("24.Q9b", True, mistakes={"que devient un biais constant après la soustraction de la moyenne du canal ?": False})
wb.record("24.Q9c", 9 / 4, decimals=2, mistakes={"la surface est divisée par 4 : écris aussi le facteur des canaux": 0.25,
          "les canaux d'entrée et de sortie sont multipliés par 3 tous les deux : C_in × C_out est multiplié par 9": 0.75,
          "la hauteur et la largeur sont divisées par 2 : la surface est divisée par 4": 9,
          "relis l'énoncé : les canaux sont multipliés par 3": 1.0})
wb.record("24.Q9d", "A", mistakes={"compare les accuracies de validation du tableau du §24.4.4": "B",
          "compare les temps par epoch du tableau du §24.4.4": "C"})
wb.record("24.Q9e", "C", mistakes={"relis la fin du §24.4.5 : les grosses couches denses portent l'essentiel des paramètres": "A",
          "relis la fin du §24.4.5 : on résume d'abord chaque canal": "B"})

WB_ANSWER {"hash": "44a71c2ab7a55263bd11a9234a726ebb378d7d54a04eb6bcc0b49fa5ddc6cdf6", "id": "24.Q9a", "kind": "str", "mistakes": {"793c6bc82cb42b1153fd805d84c698fe0cfcd25877dcbc8814810b6e0ef3d0c3": "la batchnorm normalise la sortie de la convolution", "ee382d70a86563384b6cb055895c95f4e7c7541fc993f2a6d86489c7087fa923": "relis le listing 24.44 : la convolution est écrite sans activation"}}
📝 Ex 24.Q9a : réponse enregistrée (str).
WB_ANSWER {"hash": "158cc385dc50b041da85cd6de8bcd28c836f20bb2b8330a34b61d87da4c9402b", "id": "24.Q9b", "kind": "bool", "mistakes": {"e1af28e95e89a5107a41344ee5c7988c961ba402e54ea3c0bc2c1f6abc27efff": "que devient un biais constant après la soustraction de la moyenne du canal ?"}}
📝 Ex 24.Q9b : réponse enregistrée (bool).
WB_ANSWER {"alt_coarse": ["c93942cac481c84f10e685af81f5bf08325937332d975aa7653e3f185f612998"], "decimals": 2, "hash": "e9c4704137e20e5ba021a73a895d1b26ce0ce89f3b3965795cfa6cdb4fc63966", "hash_coarse": "5ca2302f11185dda68a84033720ba9376522b04567

**24.Q10 — Séries pour un RNN : fenêtres, scaler, inverse_transform, RMSE**

In [12]:
wb.record("24.Q10a", "D", mistakes={"le scaler ne doit pas voir le test": "A", "un scaler par fenêtre changerait l'échelle d'un exemple à l'autre": "B",
          "le scaler s'ajuste sur les données d'entraînement": "C"})
wb.record("24.Q10b", "(500, 12, 1)", mistakes={"avec batch_first=True, l'axe des exemples vient en premier": "(12, 500, 1)",
          "n'oublie pas l'axe des features, même pour une seule variable": "(500, 12)"})
wb.record("24.Q10c", "A", mistakes={"une erreur se lit dans l'unité de ce qu'on prédit": "B"})
wb.record("24.Q10d", False, mistakes={"un scaler ajusté sur le test, c'est une fuite, et une autre échelle que celle apprise par le réseau": True})
wb.record("24.Q10e", "C", mistakes={"une sigmoïde écraserait la prédiction dans un intervalle": "A", "une softmax sert à des classes, pas à une valeur": "B"})
wb.record("24.Q10f", "B", mistakes={"un réseau plus gros n'est pas une référence simple": "A", "la RMSE du train ne dit rien de la prévision": "C"})

WB_ANSWER {"hash": "d1fbb22000b1c1874669bc08dac4809d3e48e9c8c0897dec0e859d2762d581e2", "id": "24.Q10a", "kind": "str", "mistakes": {"00ca648877786ddbde127857e5b1a07c57cbbf39d693bc04112d0921eda013cd": "un scaler par fenêtre changerait l'échelle d'un exemple à l'autre", "2d1fd2cba0921371d579ab9010b11f4ede8bdc0b0300afec156409d315631089": "le scaler s'ajuste sur les données d'entraînement", "9a2041c3f810ae4fcd21dcc52469d97e1e583ad50befa7df0fcbc38520fec55d": "le scaler ne doit pas voir le test"}}
📝 Ex 24.Q10a : réponse enregistrée (str).
WB_ANSWER {"hash": "ac2269e3208e93aad4ad83d32d52ea7f815ac637555966173108258e07a427d7", "id": "24.Q10b", "kind": "str", "mistakes": {"1366c94cfdad7d80bb896d6823be7c3d2deb9b9d0a3981cd1f567f65a62dbc85": "n'oublie pas l'axe des features, même pour une seule variable", "ffa4d48634bd336249c8db77c4083316cecedfe4fc0ba656a9ff25e982812e5d": "avec batch_first=True, l'axe des exemples vient en premier"}}
📝 Ex 24.Q10b : réponse enregistrée (str).
WB_ANSWER {"hash": "90f

**24.Q11 — 🕰️ return_sequences, stateful, TimeDistributed : que devient chaque option en PyTorch ?**

In [13]:
wb.record("24.Q11a", "D", mistakes={"h_n ne garde que le dernier pas": "A", "c_n est la mémoire du dernier pas": "B",
          "output[:, -1] ne garde que le dernier pas": "C"})
wb.record("24.Q11b", "A", mistakes={"h_n[0] est l'état de la première couche": "B", "c_n est la mémoire de la cellule, pas sa sortie": "C",
          "output[0] est la séquence du premier exemple": "D"})
wb.record("24.Q11c", "B", mistakes={"aplatir mélangerait tous les pas dans une seule couche dense": "A",
          "relis ce que fait nn.Linear sur un tenseur à trois dimensions": "C"})
wb.record("24.Q11d", "C", mistakes={"relis l'encadré « Le livre, corrigé » du §24.5.9 : deux exemples consécutifs d'un batch sont indépendants": "A",
          "chaque ligne a son propre état": "B"})
wb.record("24.Q11e", True, mistakes={"que devient le graphe du batch précédent après son backward() ?": False})
wb.record("24.Q11f", False, mistakes={"relis le début du §24.5.9 : de quel état part chaque séquence ?": True})

WB_ANSWER {"hash": "3fc5f467ecff5bd80562323ce3717924f8172f4fbc14671d24d1ac6a814e5907", "id": "24.Q11a", "kind": "str", "mistakes": {"59f3debe42dc3c976dc5b1b829d834ca3646bfc4a14efe5932354a901955eea0": "c_n est la mémoire du dernier pas", "a043b1008b5d824af031247d83d033885b3e7e3fc190b7ec2233bdd556949a88": "output[:, -1] ne garde que le dernier pas", "a21572d345c8061beb029a693690beab872b5402222594f1ae748957bc350f20": "h_n ne garde que le dernier pas"}}
📝 Ex 24.Q11a : réponse enregistrée (str).
WB_ANSWER {"hash": "001d8658f43eb1f249065499dedd64188bb33381b55a5da52f979a4f6f76a4b0", "id": "24.Q11b", "kind": "str", "mistakes": {"6d22d368907414b1e46155e5060b06577c70dd7fc36a11946ab0ede532d7c552": "c_n est la mémoire de la cellule, pas sa sortie", "8de0263a05cfebe3336b84db3ae4380247bacc7dbcc574894181d78c8b17a56d": "output[0] est la séquence du premier exemple", "e00013df684afda95884071c7bcb7162efe2e68692e6cdae70748960c3d69644": "h_n[0] est l'état de la première couche"}}
📝 Ex 24.Q11b : réponse en

**24.Q12 — 🕰️ API fonctionnelle de Keras 3 et forward() de PyTorch : branches, partage, gel**

In [14]:
wb.record("24.Q12a", "C", mistakes={"un module est un seul objet, avec un seul jeu de poids": "A",
          "appeler deux fois un module est permis": "B"})
wb.record("24.Q12b", "B", mistakes={"torch.cat met bout à bout, il n'additionne pas": "A", "torch.stack crée un nouvel axe": "C"})
wb.record("24.Q12c", "(4, 32, 8, 8)", mistakes={"dim=1 est l'axe des canaux": "(8, 16, 8, 8)", "torch.cat ne crée pas d'axe": "(4, 2, 16, 8, 8)",
          "torch.cat met bout à bout ; il n'additionne pas les valeurs": "(4, 16, 8, 8)"})
wb.record("24.Q12d", False, mistakes={"relis le piège classique du §24.6 : qu'est-ce qui change encore en mode train() ?": True})
wb.record("24.Q12e", "A", mistakes={"relis l'encadré « Le livre, corrigé » du §24.6.2 : le modèle se construit-il ?": "B",
          "qu'attend une convolution 2D en entrée ?": "C"})
wb.record("24.Q12f", True, mistakes={"relis l'encadré 🕰️ du §24.6": False})

WB_ANSWER {"hash": "b3296ecf1c8bb0b077590f31ab5f84e22b77a9ce12bdd2158a347f72556d0efc", "id": "24.Q12a", "kind": "str", "mistakes": {"77ff33e14d74c37fd305a1865977583f4ecde3ed18ae3395fca7cbfe6711133d": "appeler deux fois un module est permis", "f3ef1ee3083413e4ecb71205c42c45ad7eb27277bb6b525d7d44af8ba0831266": "un module est un seul objet, avec un seul jeu de poids"}}
📝 Ex 24.Q12a : réponse enregistrée (str).
WB_ANSWER {"hash": "fcf85688f22dbaea2458de910a55263031f43314f1e81e4fd8cbafbf95af527e", "id": "24.Q12b", "kind": "str", "mistakes": {"756c892bed23961736003fec427ab1918c9faf974333273c92a743eafdb56e77": "torch.cat met bout à bout, il n'additionne pas", "8c239a58d6b8933f6b70fd0c3109fab7f2e22f180b14ac810ab60406c9c26b1a": "torch.stack crée un nouvel axe"}}
📝 Ex 24.Q12b : réponse enregistrée (str).
WB_ANSWER {"hash": "d93b10acfefff99a760c9af381402e40b855552ab95794d164c69da3fff67443", "id": "24.Q12c", "kind": "str", "mistakes": {"625096111dde31ceac1303ace381845865b05364f35909b982ec5b1eacb9c

**24.R1 — Ch. 23 : Ce qu'un historique d'entraînement doit contenir pour comparer deux runs**

In [15]:
wb.record("24.R1a", "ABD", mistakes={"le test ne se regarde qu'une fois, à la fin : pas à chaque epoch": "ABDE",
          "une copie des poids à chaque epoch coûte trop de mémoire ; on garde le meilleur checkpoint": "ABCD",
          "le learning rate de chaque epoch aide à lire les courbes": "AB",
          "la loss d'entraînement fait partie de l'historique": "BD"})
wb.record("24.R1b", "BCD", mistakes={"la couleur des courbes ne distingue pas deux runs": "ABCD",
          "sans la graine, peux-tu refaire le run ?": "CD",
          "deux runs peuvent différer par un réglage laissé par défaut": "BD",
          "la durée et la machine comptent pour comparer des coûts": "BC"})
wb.record("24.R1c", False, mistakes={"relis comment train_one_epoch moyenne sa loss pendant que les poids changent": True})
wb.record("24.R1d", 5, mistakes={"relis comment les epochs sont numérotées dans l'énoncé": 4, "la meilleure epoch a la plus petite loss": 1,
          "cherche la plus petite des six valeurs": 3})

WB_ANSWER {"hash": "c04ab6c47a59bbbff10b36c81a6b5566c27c0781cfa99879214a91edbc9507ca", "id": "24.R1a", "kind": "str", "mistakes": {"231c748bcc7f1da8de343eb6b3905fcc8dcc8df05b7662d07a973c2f2197dddd": "le test ne se regarde qu'une fois, à la fin : pas à chaque epoch", "53327be919d0d000c6190e956332d779071b9073794e4db245df14637048110b": "une copie des poids à chaque epoch coûte trop de mémoire ; on garde le meilleur checkpoint", "7a72619d88c3343b47ff73e39aaec880df85d771c2bdd2d910afe695e6e91288": "le learning rate de chaque epoch aide à lire les courbes", "90c78ff0e760421c6d422a1e9b6731147a5126ce00c27ec71a74628183da4bcd": "la loss d'entraînement fait partie de l'historique"}}
📝 Ex 24.R1a : réponse enregistrée (str).
WB_ANSWER {"hash": "97cddf0c7706c26133418720b3bb8dd2ad500281b758b9790258d870a5635be0", "id": "24.R1b", "kind": "str", "mistakes": {"4f1afaaefb65ded7e3b8d2fe5f303a162ef12bf76393fb0cd9198fc8bf951866": "la durée et la machine comptent pour comparer des coûts", "6603c85ad8e838c8f18c

**24.R2 — Ch. 21 : Taille de sortie d'une convolution à padding 'same' et stride 2**

In [16]:
wb.record("24.R2a", math.ceil(28 / 2), mistakes={"avec un stride de 2, la taille est divisée par 2": 28,
          "« same » complète de zéros : la convolution ne perd pas les bords": (28 - 5) // 2 + 1})
wb.record("24.R2b", math.ceil(15 / 2), fractional="une hauteur est un nombre entier de lignes : arrondis au-dessus", mistakes={"Keras arrondit au-dessus : ⌈15/2⌉": 7})
wb.record("24.R2c", 5 // 2, mistakes={"p = ⌊k/2⌋, pas k": 5})
wb.record("24.R2d", (32 + 2 * 2 - 5) // 2 + 1, mistakes={"n'oublie pas le + 1 de la formule": 15,
          "avec un stride de 2, la sortie est deux fois plus petite": 32,
          "applique la formule avec le padding de c)": (32 - 5) // 2 + 1})
wb.record("24.R2e", (15 - 3) // 2 + 1, mistakes={"sans padding, la taille n'est pas ⌈15/2⌉": 8, "n'oublie pas le + 1 de la formule": 6,
          "le stride vaut 2 : divise par 2": 13})
wb.record("24.R2f", True, mistakes={"relis le ch. 21 : PyTorch accepte-t-il un noyau pair avec padding=\"same\" ?": False})

WB_ANSWER {"hash": "e71ffba34cbf0d197a09298b44d72b0336a32f660475b81d857294cb20e3654b", "id": "24.R2a", "kind": "int", "magnitude": 1, "mistakes": {"5d6866e5ffd42a5f5007657cd548ba1c408776f720fee0e7eb7341a981e8ec65": "avec un stride de 2, la taille est divisée par 2", "843f07f2ee836b149bc20a7b3bff4de23412ca430f50fddf7cb47d496230b06a": "« same » complète de zéros : la convolution ne perd pas les bords"}, "sign": 1}
📝 Ex 24.R2a : réponse enregistrée (int).
WB_ANSWER {"fractional": "une hauteur est un nombre entier de lignes : arrondis au-dessus", "hash": "fa31566afd527a9b6245ceee406727590201da2ef2ffae78bb3ab4cb56d8400b", "id": "24.R2b", "kind": "int", "magnitude": 0, "mistakes": {"aad8ea55e191370ce95a889e4f12d38c334d716e5368ad1988518bd962c3f5ee": "Keras arrondit au-dessus : ⌈15/2⌉"}, "sign": 1}
📝 Ex 24.R2b : réponse enregistrée (int).
WB_ANSWER {"hash": "aec7fb9c81fae9b8e86de541b4dc0482473bda2aac32ead4168f87d7db3c5448", "id": "24.R2c", "kind": "int", "magnitude": 0, "mistakes": {"24c1a4bdb

**24.R3 — Ch. 17 : Ce que change la division des logits avant le softmax**

In [17]:
wb.record("24.R3a", softmax(Z_R3).max(), decimals=3, mistakes={"la softmax prend des exponentielles, pas les logits eux-mêmes": 2 / 3,
          "c'est la plus grande probabilité qu'on demande": softmax(Z_R3).min()})
wb.record("24.R3b", softmax(Z_R3 / 0.5).max(), decimals=3, mistakes={"pour T = 0,5, diviser par T revient à doubler les logits": softmax(Z_R3 * 0.5).max(),
          "applique la température : divise chaque logit par T avant la softmax": softmax(Z_R3).max()})
wb.record("24.R3c", softmax(Z_R3 / 2).max(), decimals=2, mistakes={"pour T = 2, diviser par T revient à diviser les logits par 2": softmax(Z_R3 * 2).max(),
          "applique la température : divise chaque logit par T avant la softmax": softmax(Z_R3).max()})
wb.record("24.R3d", "B", mistakes={"c'est quand T grandit que la distribution s'aplatit": "A",
          "regarde ce que deviennent les écarts entre logits divisés par un T minuscule": "C"})
wb.record("24.R3e", True, mistakes={"diviser par un nombre positif change-t-il l'ordre des logits ?": False})

WB_ANSWER {"decimals": 3, "hash": "abde04c127e90193cba3d87bee4324d6cdadc188f896256134314ba479653153", "hash_coarse": "9679bd026352abfa8c0d348ebb279990ebe39450bf56f9508ad1d501ad092838", "id": "24.R3a", "kind": "float", "magnitude": -1, "mistakes": {"e5d9ba73b178ec762ff4ed6d7a735f6d4f15bec9dc2e2cc2125376aac57b1a5f": "c'est la plus grande probabilité qu'on demande", "ed9e29e4eca38eebb2fd18ae9b7db59036986b5199fbe41a14bdc76ccac29015": "la softmax prend des exponentielles, pas les logits eux-mêmes"}, "sign": 1}
📝 Ex 24.R3a : réponse enregistrée (float, 3 décimale(s)).
WB_ANSWER {"decimals": 3, "hash": "972edbf49469eeef913a1f08a9c66a66e8be8811184064eb7a8bd6faf3b1b232", "hash_coarse": "c582e796b55c8dc662d43677316e5ee13e08728a050139bc9bce9da9fce2a7e7", "id": "24.R3b", "kind": "float", "magnitude": -1, "mistakes": {"3eb7e68c130b95cdf9080d76dd4c884d1bb0909711a71927aa644fd2ef0191a8": "applique la température : divise chaque logit par T avant la softmax", "e9d7cc5ed262f9bbcf417bb29bd0fea04acefabf1a

**Ex 24.1 — Le coût d'une grille (27 hyperparamètres, 54 entraînements) contre une recherche aléatoire : probabilité de tomber dans les 5 % meilleurs réglages en n essais**

In [18]:
wb.record("24.1a", 2 ** 27, mistakes={"2 valeurs pour chacun des 27 : 2 puissance 27, pas 27 au carré": 27 ** 2,
          "les nombres de valeurs se multiplient, ils ne s'additionnent pas": 54})
wb.record("24.1b", 2 ** 27 / (3600 * 24 * 365), decimals=1, mistakes={"une année compte 3 600 × 24 × 365 secondes": 2 ** 27 / (3600 * 24)})
wb.record("24.1c", 3 * (3 * 3 * 2), mistakes={"chaque candidat est entraîné une fois par fold": 18, "multiplie les nombres de valeurs": 3 * (3 + 3 + 2)})
wb.record("24.1d", 54 * 70 // 60, mistakes={"en minutes, pas en secondes": 54 * 70, "chaque candidat est entraîné une fois par fold": 18 * 70 // 60})
wb.record("24.1e", 1 - 0.95 ** 20, decimals=2, mistakes={"c'est la probabilité de tout rater ; on demande celle d'en trouver au moins un": 0.95 ** 20,
          "0,05 × 20 compterait deux fois les tirages qui réussissent ensemble": 1.0})
wb.record("24.1f", math.ceil(math.log(0.05) / math.log(0.95)), mistakes={"arrondis au-dessus : le premier entier n tel que 0,95ⁿ ≤ 0,05": 58,
          "pose 0,95ⁿ ≤ 0,05, puis prends les logarithmes": 20},
          fractional="un nombre de tirages est entier : prends le premier entier qui suffit")
wb.record("24.1g", 1 - 0.95 ** 18, decimals=3, mistakes={"c'est la probabilité de tout rater ; on demande celle d'en trouver au moins un": 0.95 ** 18,
          "tu as pris le mauvais nombre de tirages": 1 - 0.95 ** 20})

WB_ANSWER {"hash": "16dd041cce398fc13053510d4062ecb76b1bab394dcac5da2c95455b87be3048", "id": "24.1a", "kind": "int", "magnitude": 8, "mistakes": {"906b38815db239f635808ca717e3c1eb4d93582486da63068d128c1b0adc4109": "les nombres de valeurs se multiplient, ils ne s'additionnent pas", "f8aa13d4ea9a289bfc7c402d79cf21fc3c856a9073977bf2867a9d03c4d0d321": "2 valeurs pour chacun des 27 : 2 puissance 27, pas 27 au carré"}, "sign": 1}
📝 Ex 24.1a : réponse enregistrée (int).
WB_ANSWER {"decimals": 1, "hash": "1de2873c6f787d2ec39e7a6beeae5be34487479340415a7597ab3bc5f479ba7f", "hash_coarse": "6294d97206194244d3bfd7eb91ff8e7951df039dda88e4c560fee4fd66de4168", "id": "24.1b", "kind": "float", "magnitude": 0, "mistakes": {"efce45d74a1777858e4f7f4400b97fc7d975613f23045574e2a1d559e1b49835": "une année compte 3 600 × 24 × 365 secondes"}, "sign": 1}
📝 Ex 24.1b : réponse enregistrée (float, 1 décimale(s)).
WB_ANSWER {"hash": "4c3c4190240c35baf17bc7174b19c6059e18303c2f5e68d37e804f04d3529736", "id": "24.1c", "

**Ex 24.2 — Formes et paramètres du CNN à pooling du livre**

In [19]:
wb.record("24.2a", "(256, 30, 28, 28)", mistakes={"avec padding=2, un noyau de 5 garde la taille": "(256, 30, 24, 24)",
          "PyTorch range les canaux juste après l'axe des exemples": "(256, 28, 28, 30)",
          "n'oublie pas l'axe des exemples": "(30, 28, 28)"})
wb.record("24.2b", "(256, 30, 14, 14)", mistakes={"le pooling 2 × 2 divise la hauteur et la largeur par 2": "(256, 30, 28, 28)",
          "le pooling ne change pas le nombre de canaux": "(256, 15, 14, 14)",
          "n'oublie pas l'axe des exemples": "(30, 14, 14)"})
wb.record("24.2c", P_CONV1, mistakes={"n'oublie pas les 30 biais": 30 * 25, "un filtre 5 × 5 a 25 poids, pas 5": 30 * (5 + 1)})
wb.record("24.2d", P_CONV2, mistakes={"n'oublie pas les 16 biais": 16 * 30 * 9, "chaque filtre a autant de canaux que l'entrée (30)": 16 * (9 + 1),
          "un filtre 3 × 3 a 9 poids par canal": 16 * (30 * 3 + 1)})
wb.record("24.2e", N_FLAT, mistakes={"compte aussi les canaux : 16 valeurs par position": 49,
          "la sortie du second pooling est en 7 × 7, pas en 14 × 14": 16 * 14 * 14, "la seconde convolution a 16 filtres, pas 30": 30 * 7 * 7})
wb.record("24.2f", dense(N_FLAT, 128), mistakes={"n'oublie pas les 128 biais": N_FLAT * 128, "la couche reçoit 16 × 7 × 7 valeurs": dense(49, 128)})
wb.record("24.2g", P_TOTAL, mistakes={"n'oublie pas les deux dernières couches denses": P_CONV1 + P_CONV2 + dense(N_FLAT, 128),
          "n'oublie pas les deux convolutions": P_DENSE,
          "la première couche dense reçoit 16 × 7 × 7 valeurs": P_CONV1 + P_CONV2 + dense(49, 128) + dense(128, 64) + dense(64, 10)})
wb.record("24.2h", P_DENSE / P_TOTAL, decimals=3, mistakes={"relis quelle part est demandée": (P_CONV1 + P_CONV2) / P_TOTAL,
          "une fraction entre 0 et 1, pas un pourcentage": 100 * P_DENSE / P_TOTAL,
          "les trois couches denses, pas seulement la première": dense(N_FLAT, 128) / P_TOTAL})

WB_ANSWER {"hash": "8d613b9427778519f677795cb45a6826ecfae2f7c350e23d4295880a7026bd03", "id": "24.2a", "kind": "str", "mistakes": {"2e94e95950a4432ea373e7fe5510d3f3cc66c784298b29e6b0024ff4f704fa59": "n'oublie pas l'axe des exemples", "4771ebec9131f58c3c0253c6733808ca407b828138b3017a10afb92c7872c225": "PyTorch range les canaux juste après l'axe des exemples", "91681555fcd0e577cc62b24dbd3ada0c4487d86ae7caca9cc88d391353c5b581": "avec padding=2, un noyau de 5 garde la taille"}}
📝 Ex 24.2a : réponse enregistrée (str).
WB_ANSWER {"hash": "d9dbc8cbc95985da7448de027e4c2d966f8c567b41f9f0b4d11c63c3520587fe", "id": "24.2b", "kind": "str", "mistakes": {"99af80c5d99136174eb1b2ec5afb18e6f1ee2ba91a3b519bf83355f0d4b960ae": "le pooling 2 × 2 divise la hauteur et la largeur par 2", "b40565d738ea8a2b5f38bc6ce7184f40ffaf87f079827db46a6c89cdfd057d07": "n'oublie pas l'axe des exemples", "dee76760c97696b82ad356dc69bf2517f9c531cbce9f8737efaa11bfb6fce029": "le pooling ne change pas le nombre de canaux"}}
📝 Ex 2

**Ex 24.3 — Stride ou pooling : compter les multiplications**

In [20]:
wb.record("24.3a", conv_mac(28, 28, 30, 1, 5), mistakes={"un noyau 5 × 5 compte 25 multiplications par position": conv_mac(28, 28, 30, 1, 1) * 5,
          "avec padding=2, la sortie est en 28 × 28": conv_mac(24, 24, 30, 1, 5)})
wb.record("24.3b", conv_mac(14, 14, 16, 30, 3), mistakes={"la seconde convolution travaille sur la sortie du premier pooling, en 14 × 14": conv_mac(28, 28, 16, 30, 3),
          "chaque filtre a 30 canaux d'entrée": conv_mac(14, 14, 16, 1, 3),
          "le pooling vient après la convolution : sa sortie est en 14 × 14, pas en 7 × 7": conv_mac(7, 7, 16, 30, 3)})
wb.record("24.3c", MAC_DENSE, mistakes={"les trois couches denses : 784 → 128 → 64 → 10": 784 * 128,
          "l'énoncé néglige les biais": 784 * 128 + 128 + 128 * 64 + 64 + 64 * 10 + 10,
          "la première couche dense reçoit 784 valeurs": 49 * 128 + 128 * 64 + 64 * 10})
wb.record("24.3d", MAC_STRIDE, mistakes={"avec un stride de 2, la seconde convolution sort en 7 × 7": conv_mac(14, 14, 30, 1, 5) + conv_mac(14, 14, 16, 30, 3),
          "les deux convolutions ensemble": conv_mac(14, 14, 30, 1, 5),
          "additionne les deux convolutions": conv_mac(7, 7, 16, 30, 3),
          "avec un stride de 2, la première convolution sort en 14 × 14": conv_mac(28, 28, 30, 1, 5) + conv_mac(7, 7, 16, 30, 3)})
wb.record("24.3e", (MAC_POOL + MAC_DENSE) / (MAC_STRIDE + MAC_DENSE), decimals=2, mistakes={"ajoute les couches denses aux deux versions": MAC_POOL / MAC_STRIDE,
          "la version à pooling au numérateur": (MAC_STRIDE + MAC_DENSE) / (MAC_POOL + MAC_DENSE)})
wb.record("24.3f", 30 * 14 * 14 * 3, mistakes={"3 comparaisons par valeur de sortie, pas par valeur d'entrée": 30 * 28 * 28 * 3,
          "compte les comparaisons, pas seulement les valeurs de sortie": 30 * 14 * 14,
          "n'oublie pas les 30 canaux": 14 * 14 * 3})
wb.record("24.3g", 147 / 36, decimals=1, mistakes={"le rapport de la version à pooling à la version à stride": 36 / 147})

WB_ANSWER {"hash": "fe9a37b1d302fb940fdc04e66a0920b51ca301e359d280af064d0cbf0da97daa", "id": "24.3a", "kind": "int", "magnitude": 5, "mistakes": {"4636068a727fc06c3a57517d93f68f1a7f0f17542ac00f50f6edb80f4205f183": "avec padding=2, la sortie est en 28 × 28", "46a283701fe5e8398065d090385be2c3080f6d7b7c61635e80b3fe6c7a5679f0": "un noyau 5 × 5 compte 25 multiplications par position"}, "sign": 1}
📝 Ex 24.3a : réponse enregistrée (int).
WB_ANSWER {"hash": "457330169657ac65baff7dbd65c44619f8fbf6ee698ec6a8a23045cfb4d316cd", "id": "24.3b", "kind": "int", "magnitude": 5, "mistakes": {"711932a0a363a4812c39052eb7cee3f8061b16633139fb8f36158a9493e1477a": "chaque filtre a 30 canaux d'entrée", "9b0af7b02dc044c1978f1fc69169d8c635eca9dc103e2df1170e588d4e8694e1": "la seconde convolution travaille sur la sortie du premier pooling, en 14 × 14", "e6e72e804df8551a20b1ecf5b5049e926d84d2a617435e20d4a9a1646df099dc": "le pooling vient après la convolution : sa sortie est en 14 × 14, pas en 7 × 7"}, "sign": 1}
📝 

**Ex 24.4 — Ce que renvoie nn.LSTM : output, h_n et c_n**

In [21]:
wb.record("24.4a", "(8, 5, 4)", mistakes={"output a H valeurs par pas, pas D": "(8, 5, 3)", "avec batch_first=True, l'axe des exemples vient en premier": "(5, 8, 4)",
          "output garde tous les pas de temps": "(8, 4)"})
wb.record("24.4b", "(2, 8, 4)", mistakes={"batch_first ne change pas la forme de h_n": "(8, 2, 4)", "h_n a une ligne par couche": "(1, 8, 4)",
          "h_n a H valeurs par exemple": "(2, 8, 3)"})
wb.record("24.4c", "A", mistakes={"output[0] est la séquence du premier exemple : écris sa forme": "B",
          "c_n a la forme de h_n tout entier, pas celle d'une de ses lignes": "C",
          "output[:, :, 0] garde tous les pas de temps : écris sa forme": "D"})
wb.record("24.4d", "(8, 5, 8)", mistakes={"les deux sens sont côte à côte sur la dernière dimension": "(8, 10, 4)",
          "avec deux sens, la dernière dimension double": "(8, 5, 4)"})
wb.record("24.4e", "(4, 8, 4)", mistakes={"une ligne par couche et par sens": "(2, 8, 4)", "les sens multiplient le nombre de lignes, pas H": "(2, 8, 8)"})
wb.record("24.4f", lstm_layer_params(3, 4) + lstm_layer_params(4, 4), mistakes={"PyTorch range deux vecteurs de biais par couche": 4 * 4 * (3 + 4) + 16 + 4 * 4 * (4 + 4) + 16,
          "la seconde couche reçoit H = 4 valeurs, pas D = 3": 2 * lstm_layer_params(3, 4),
          "un LSTM a quatre blocs de poids (les trois portes et le candidat)": (4 * (3 + 4) + 8) + (4 * (4 + 4) + 8),
          "n'oublie pas la seconde couche": lstm_layer_params(3, 4),
          "n'oublie pas la première couche": lstm_layer_params(4, 4)})
wb.record("24.4g", "(8, 5, 1)", mistakes={"nn.Linear remplace la dernière dimension seulement": "(8, 1)", "nn.Linear agit sur la dernière dimension": "(8, 5, 4)"})
wb.record("24.4h", "(5, 8, 3)", mistakes={"sans batch_first, le temps vient en premier": "(8, 5, 3)", "les features restent sur la dernière dimension": "(3, 5, 8)"})

WB_ANSWER {"hash": "f7907cc11c64ca650dd88a086b5cf447ba3b315477310447a337c7a9b2f4f28d", "id": "24.4a", "kind": "str", "mistakes": {"4c2ef37cc919a6a92c018a35c391fe7d13776679cebbb19005814ff1ec7a0e34": "output a H valeurs par pas, pas D", "ac3d75493f6d4264bda617062b5ac5ec5faf5f1edc1cf2e54f06f77fe77dc312": "avec batch_first=True, l'axe des exemples vient en premier", "f17558945f7ba5818a9116cae536afaf7a76d636a85eb45d6b431f41a835ca18": "output garde tous les pas de temps"}}
📝 Ex 24.4a : réponse enregistrée (str).
WB_ANSWER {"hash": "51ac475becc4ff41429772ded7ae329a8076a8f7633bbbd4a1d3e561de765a2a", "id": "24.4b", "kind": "str", "mistakes": {"1d3cf548bb15cc7d6023beea2f00f99bdfe9a3e0941bad18f0edbc97931dd4fa": "h_n a H valeurs par exemple", "2f07a14e3cc1a4bcc939533300c13b1343dd7f44eca8346c0114423912a57015": "batch_first ne change pas la forme de h_n", "60c40fb78ab4fe7ad75738a8ddf6758228b5acea7fbeb5b6b053c6fd4d6ea5fa": "h_n a une ligne par couche"}}
📝 Ex 24.4b : réponse enregistrée (str).
WB_ANSW

**Ex 24.5 — Fenêtres, pas d'optimisation et durée : de 200 à 20 000 points**

In [22]:
wb.record("24.5a", 200 - 13, mistakes={"la dernière fenêtre doit laisser une valeur pour sa cible": 200 - 13 + 1,
          "les fenêtres commencent à chaque pas : elles se chevauchent": 200 // 13})
wb.record("24.5b", 20_000 - 13, mistakes={"la dernière fenêtre doit laisser une valeur pour sa cible": 20_000 - 13 + 1,
          "les fenêtres commencent à chaque pas : elles se chevauchent": 20_000 // 13})
wb.record("24.5c", 20_000 - 13, mistakes={"un pas par exemple : compte les fenêtres, pas les points": 20_000})
wb.record("24.5d", math.ceil((20_000 - 13) / 32), fractional="un nombre de pas est entier : le dernier mini-batch, incomplet, compte aussi",
          mistakes={"arrondis au-dessus : le dernier mini-batch, incomplet, compte aussi": (20_000 - 13) // 32})
wb.record("24.5e", 100 * 5 / 60, decimals=1, mistakes={"en minutes, pas en secondes": 500, "500 s font 8 min 20 s, soit 8,33 min et non 8,20": 8.2})
wb.record("24.5f", 28 * 3600 // 100, mistakes={"28 heures pour 100 epochs : divise par 100": 28 * 3600, "en secondes : une heure fait 3 600 s": 28 * 60 // 100})
wb.record("24.5g", 5 * (200 - 13) / (20_000 - 13), decimals=3, mistakes={"proportionnelle au nombre d'exemples, pas au nombre de points": 5 * 200 / 20_000,
          "la petite série a moins d'exemples : l'epoch dure moins de 5 s": 5 * (20_000 - 13) / (200 - 13)})

WB_ANSWER {"hash": "5a75a295a080e8d591953035db9c137f54d9ebd63bbc535f751bf7cf8a07f83b", "id": "24.5a", "kind": "int", "magnitude": 2, "mistakes": {"61dc4ef84b3d6ebaa082036f20a9c46f552beff8d3be327431ad0d64d8d49db2": "la dernière fenêtre doit laisser une valeur pour sa cible", "a25dc020bee603de2858ceee21b11815247f5ec57192ee7e44de3d12cefed705": "les fenêtres commencent à chaque pas : elles se chevauchent"}, "sign": 1}
📝 Ex 24.5a : réponse enregistrée (int).
WB_ANSWER {"hash": "6b4ed754e66795f26e29eddd0e5399ede5f59c2c8035a9bc21448f0e98b9ca7d", "id": "24.5b", "kind": "int", "magnitude": 4, "mistakes": {"8407b4c1e2af4e1445b17bbd963ba7138d257b9b4815e374dbb841dce0f7dc0f": "les fenêtres commencent à chaque pas : elles se chevauchent", "f3ea8305913ad4978bea72cd69190eff709970d3ac23be576f3a56cad2a51dec": "la dernière fenêtre doit laisser une valeur pour sa cible"}, "sign": 1}
📝 Ex 24.5b : réponse enregistrée (int).
WB_ANSWER {"hash": "5a90d74c9e6fffa05a515fe028b89529e9b111a635e8cb33aca628cfac5ae7c0

**Ex 24.7 — Fermi : le budget d'une recherche d'hyperparamètres sur CPU, sur GPU et en précision mixte**

In [23]:
wb.record("24.7a", round(OPS_247 / 1e12), mistakes={"compte 2 opérations par MAC": round(OPS_247 / 2 / 1e12),
          "un entraînement coûte 3 fois la passe avant": round(OPS_247 / 3 / 1e12),
          "en milliers de milliards (10¹²), pas en milliards": round(OPS_247 / 1e9),
          "n'oublie pas les 20 epochs": round(OPS_247 / 20 / 1e12)})
wb.record("24.7b", round(CPU_MIN_247), mistakes={"en minutes, pas en secondes": round(CPU_MIN_247 * 60), "en minutes, pas en heures": round(CPU_MIN_247 / 60),
          "un entraînement coûte 3 fois la passe avant": round(CPU_MIN_247 / 3)})
wb.record("24.7c", 4 * 3 * 3 * 3, mistakes={"chaque candidat est entraîné une fois par fold": 36, "multiplie les nombres de valeurs": (4 + 3 + 3) * 3})
wb.record("24.7d", round(108 * CPU_MIN_247 / 60), mistakes={"en heures, pas en minutes": round(108 * CPU_MIN_247), "n'oublie pas les 3 folds": round(36 * CPU_MIN_247 / 60)})
wb.record("24.7e", 108 * OPS_247 / 1e13 / 3600, decimals=2, mistakes={"en heures, pas en secondes": 108 * OPS_247 / 1e13, "en heures, pas en minutes": 108 * OPS_247 / 1e13 / 60})
wb.record("24.7f", 108 * OPS_247 / 1e13 / 2.5 / 60, decimals=1, mistakes={"2,5 fois plus vite : divise la durée, ne la multiplie pas": 108 * OPS_247 / 1e13 * 2.5 / 60,
          "en minutes, pas en heures": 108 * OPS_247 / 1e13 / 2.5 / 3600,
          "648 s font 10 min 48 s, soit 10,8 min": 6.5})
wb.record("24.7g", round(30 * CPU_MIN_247 / 60), mistakes={"en heures, pas en minutes": round(30 * CPU_MIN_247),
          "un seul jeu de validation : un entraînement par réglage, pas trois": round(90 * CPU_MIN_247 / 60)})

WB_ANSWER {"hash": "e915a2887fbcec98d68943e0cd63d020f1145bb43b0f398097fe5d1c32b067e2", "id": "24.7a", "kind": "int", "magnitude": 2, "mistakes": {"3419354f0eadcf74970f5aebdf5c6efa1066fb8438afb4c016d1df374fabd2c1": "n'oublie pas les 20 epochs", "9d19566ea5ec0b0f63e4db1e9539f12d0779ef788a6135e1d274087efcb3a6ca": "un entraînement coûte 3 fois la passe avant", "be0e8bb1e78a7499a2ed3dc79602e52f64b7b9847eb72311a69efb1b65cd0975": "en milliers de milliards (10¹²), pas en milliards", "d442f7806bff44954bfad1a03011dcb1a59df24f7a4410a7d4474cb3d4c1fd96": "compte 2 opérations par MAC"}, "sign": 1}
📝 Ex 24.7a : réponse enregistrée (int).
WB_ANSWER {"hash": "e7f570db0a0a5b5b4791c806ba287e4dbbca455388d584e769c0376a2caa3247", "id": "24.7b", "kind": "int", "magnitude": 1, "mistakes": {"7054ba172e95da2aa5e55bae104894db6beb9ab4e051837c7a1bd4d4263ea853": "en minutes, pas en heures", "795f22d530036879723e4b0dcffa7cca122a610f094864c27404c809ec2a7684": "en minutes, pas en secondes", "d6c4294d3511f356971757eb1c

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Sais-tu mener une amélioration par expériences (un réglage à la fois, un journal) et chiffrer le coût d'une recherche d'hyperparamètres ?
2. Sais-tu envelopper un réseau PyTorch dans un estimateur scikit-learn et le valider sans fuite ?
3. Sais-tu construire un CNN et un LSTM en PyTorch, lire leurs formes, et écrire un bloc résiduel dans `forward()` ?

**Pour aller plus loin** : le « Deep Learning Tuning Playbook » de Google et l'article de He et coll. (2016), cités dans la fiche. La suite : le checkpoint de la partie V, puis le ch. 25, les autoencodeurs.

*Les exercices de code 24.11 à 24.31 (parties A à E : améliorer un MLP, scikit-learn, les CNN sur CIFAR-10, les séries et le LSTM, les architectures à branches, la précision mixte, la génération de texte et le défi) seront ajoutés à la prochaine session de génération ; relance alors `python tools/start_chapter.py 24` pour obtenir le notebook complet.*